# Evaluation (precision, recall, f1-score - Micro, Macro, Weighted)

## Get entities LLM

In [ ]:
import pandas as pd
import numpy as np
import ast
import json

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    mean_absolute_error,
    precision_score,
    recall_score,
    classification_report,
    hamming_loss,
    jaccard_score,
    precision_recall_fscore_support,
)

from get_entities_LLM import extract_llm_entities, entity_vocab
from get_entities_ft_nlp import extract_ft_entities, ENTITY_LIST
from get_sentiment_nlp import extract_nlp_sentiment
from get_sentiment_llm import extract_llm_sentiment

from tqdm import tqdm

/opt/anaconda3/envs/arp_feds/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]


[DEBUG] Loaded optimal threshold: 0.4
[DEBUG] Config loaded:
  FINETUNED_DIR: ./finbert-finetuned
  BASE_MODEL_NAME: yiyanghkust/finbert-tone
  DEF_WINDOW: 8
  DECISION_THRESHOLD: 0.4
[DEBUG] Label mapping loaded: 7 mappings
  Score ranges: -1.00 to 1.00
  Label indices: [0, 1, 2, 3, 4, 5, 6]
[DEBUG] Custom entities loaded: 42 canonical entities
  Total aliases: 229
  Sample canonicals: ['Federal Reserve', 'Interest Rates', 'Inflation']
[DEBUG] Tokenizer source selected: ./finbert-finetuned
  Fine-tuned tokenizer files found: True
[DEBUG] Device: cpu
[DEBUG] Loading model with 7 labels
[DEBUG] Building CORAL model with 7 labels
[DEBUG] Loading base encoder from: yiyanghkust/finbert-tone
[DEBUG] Encoder config - hidden_size: 768, dropout: 0.1
[DEBUG] CORAL model created with 6 thresholds
[DEBUG] Looking for model files in: ./finbert-finetuned
[DEBUG] Files found: ['model.safetensors', 'tokenizer_config.json', 'special_tokens_map.json', 'tokenizer.json', 'training_args.bin', 'vocab.txt']

In [ ]:
# Load your labeled dataset
df = pd.read_csv("data/ARP_dataset_fixed_Sentiment.csv")
df["entities"] = df["Entities"].apply(eval)

# Split into train and test
train_df, test_df = train_test_split(df, test_size=0.3, random_state=123)

# get sentences and true entities to list
## sentences
test_sentences = test_df["Sentence"].tolist()
## entities
true_entities_list_with_sentiment = test_df["entities"].tolist()
true_entities_list = [[ent[0] for ent in entity_group if ent[0] != ''] for entity_group in true_entities_list_with_sentiment]


### boost

In [ ]:
from tqdm import tqdm

BATCH_SIZE = 200 

def extract_llm_entities_in_batches(sentences, batch_size=BATCH_SIZE):
    results = []
    n = len(sentences)
    for start in tqdm(range(0, n, batch_size), desc="LLM Prediction"):
        batch = sentences[start:start+batch_size]
        results.extend(extract_llm_entities(batch))
    return results

pred_results = extract_llm_entities_in_batches(test_sentences)
pred_entities_list = [res["entities"] for res in pred_results]

LLM Prediction:   0%|          | 0/3 [00:00<?, ?it/s]2025-08-31 14:10:30,545 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-31 14:10:30,546 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-31 14:10:30,546 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-31 14:10:30,546 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-31 14:10:30,547 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-31 14:10:30,547 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-31 14:10:30,547 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-31 14:10:30,547 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-31 14:10:30,548 | INFO | HTTP Request: POST

### evaluate

In [ ]:
# Convert to multi-hot
def entities_to_multihot(entities, vocab):
    multihot = [0] * len(vocab)
    for ent in entities:
        if ent in vocab:
            multihot[vocab.index(ent)] = 1
    return multihot

y_true = [entities_to_multihot(ents, entity_vocab) for ents in true_entities_list]
y_pred = [entities_to_multihot(ents, entity_vocab) for ents in pred_entities_list]


In [5]:
y_true_bin = np.asarray(y_true).astype(int)
y_pred_bin = np.asarray(y_pred).astype(int)

# F1
print("Micro F1:",    f1_score(y_true_bin, y_pred_bin, average="micro",    zero_division=0))
print("Macro F1:",    f1_score(y_true_bin, y_pred_bin, average="macro",    zero_division=0))
print("Weighted F1:", f1_score(y_true_bin, y_pred_bin, average="weighted", zero_division=0))

# Accuracy
hamming_acc = 1.0 - hamming_loss(y_true_bin, y_pred_bin)                                    # Hamming accuracy
jaccard_acc = jaccard_score(y_true_bin, y_pred_bin, average="samples", zero_division=0)     # Example-based Jaccard
subset_acc  = accuracy_score(y_true_bin, y_pred_bin)                                        # Subset accuracy

print("\nAccuracy metrics (multi-label):")
print(f"Hamming accuracy:            {hamming_acc:.4f}")
print(f"Example-based Jaccard acc.:  {jaccard_acc:.4f}")
print(f"Subset accuracy (exact match): {subset_acc:.4f}")

print("\nClassification Report:")
print(classification_report(
    y_true_bin, y_pred_bin,
    target_names=entity_vocab,
    zero_division=0
))


Micro F1: 0.6559571619812584
Macro F1: 0.5636122069057381
Weighted F1: 0.6528697889491766

Accuracy metrics (multi-label):
Hamming accuracy:            0.9695
Example-based Jaccard acc.:  0.4000
Subset accuracy (exact match): 0.3367

Classification Report:
                           precision    recall  f1-score   support

          Federal Reserve       0.85      0.67      0.75        87
           Interest Rates       0.74      0.56      0.64        50
                Inflation       0.98      0.88      0.93        93
               Employment       0.84      0.87      0.86        31
             Unemployment       1.00      1.00      1.00         8
                      GDP       0.55      0.52      0.53        31
                    Trade       0.50      1.00      0.67         3
                 Congress       0.75      1.00      0.86         3
          Monetary Policy       0.65      0.74      0.70        66
      Financial Stability       0.25      1.00      0.40         1
     

In [ ]:
# --- Per-entity metrics ---

N, L = y_true_bin.shape
rows = []
for i, name in enumerate(entity_vocab):
    yt = y_true_bin[:, i]
    yp = y_pred_bin[:, i]

    tp = int(np.sum((yt == 1) & (yp == 1)))
    tn = int(np.sum((yt == 0) & (yp == 0)))
    fp = int(np.sum((yt == 0) & (yp == 1)))
    fn = int(np.sum((yt == 1) & (yp == 0)))

    # per-entity Accuracy（Hamming accuracy）
    acc_i = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else np.nan

    # per-entity Jaccard（binary）
    jac_i = jaccard_score(yt, yp, average="binary", zero_division=0)

    # Precision / Recall / F1（binary）
    prec_i, rec_i, f1_i, supp_pos = precision_recall_fscore_support(
        yt, yp, average="binary", zero_division=0
    )

    rows.append({
        "entity": name,
        "support_total": int(len(yt)),
        "support_pos": int(np.sum(yt == 1)),
        "prevalence": float(np.mean(yt == 1)),
        "predicted_pos": int(np.sum(yp == 1)),
        "pred_rate": float(np.mean(yp == 1)),
        "accuracy": float(acc_i),
        "jaccard": float(jac_i),
        "precision": float(prec_i),
        "recall": float(rec_i),
        "f1": float(f1_i),
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
    })

per_entity_df = pd.DataFrame(rows)


per_entity_df = per_entity_df.sort_values(["support_pos", "jaccard"], ascending=[False, False])

print("\nPer-entity metrics (support_pos then jaccard):")
print(per_entity_df.to_string(index=False))

# Across entities
macro_label_acc = per_entity_df["accuracy"].mean()
macro_label_jac = per_entity_df["jaccard"].mean()
macro_label_f1  = per_entity_df["f1"].mean()

print("\nMacro (over entities):")
print(f"Macro label accuracy: {macro_label_acc:.4f}")
print(f"Macro label Jaccard:  {macro_label_jac:.4f}")
print(f"Macro label F1:       {macro_label_f1:.4f}")


Per-entity metrics (support_pos then jaccard):
                   entity  support_total  support_pos  prevalence  predicted_pos  pred_rate  accuracy  jaccard  precision   recall       f1  TP  FP  FN  TN
         Economic Outlook            401          121    0.301746             52   0.129676  0.788030 0.341085   0.846154 0.363636 0.508671  44   8  77 272
                Inflation            401           93    0.231920             84   0.209476  0.967581 0.863158   0.976190 0.881720 0.926554  82   2  11 306
          Federal Reserve            401           87    0.216958             68   0.169576  0.902743 0.597938   0.852941 0.666667 0.748387  58  10  29 304
          Monetary Policy            401           66    0.164589             75   0.187032  0.892768 0.532609   0.653333 0.742424 0.695035  49  26  17 309
           Interest Rates            401           50    0.124688             38   0.094763  0.920200 0.466667   0.736842 0.560000 0.636364  28  10  22 341
                

## Get entities nlp

In [ ]:
# Load your labeled dataset
df = pd.read_csv("data/ARP_dataset_fixed_Sentiment.csv")
df["entities"] = df["Entities"].apply(eval)

# Split into train and test
train_df, test_df = train_test_split(df, test_size=0.3, random_state=123)

# get sentences and true entities to list
## sentences
test_sentences = test_df["Sentence"].tolist()
## entities
true_entities_list_with_sentiment = test_df["entities"].tolist()
true_entities_list = [[ent[0] for ent in entity_group if ent[0] != ''] for entity_group in true_entities_list_with_sentiment]

### ft nlp

In [ ]:
#%%
# Wrap original extract_ft_nlp_entities 
def extract_ft_nlp_entities_with_progress(sentences):
    results = []
    for sent in tqdm(sentences, desc="Finetune NLP Prediction"):
        result = extract_ft_entities([sent])[0]
        results.append(result)
    return results

#%%
# Run prediction with progress bar
pred_results = extract_ft_nlp_entities_with_progress(test_sentences)
pred_entities_list = [res["entities"] for res in pred_results]

Finetune NLP Prediction: 100%|██████████| 401/401 [00:31<00:00, 12.63it/s]


In [9]:
#%%
# Convert to multi-hot
def entities_to_multihot(entities, vocab):
    multihot = [0] * len(vocab)
    for ent in entities:
        if ent in vocab:
            multihot[vocab.index(ent)] = 1
    return multihot

y_true = [entities_to_multihot(ents, entity_vocab) for ents in true_entities_list]
y_pred = [entities_to_multihot(ents, entity_vocab) for ents in pred_entities_list]

from sklearn.metrics import (
    f1_score, classification_report,
    hamming_loss, jaccard_score,
    accuracy_score,
)
import numpy as np

y_true_bin = np.asarray(y_true).astype(int)
y_pred_bin = np.asarray(y_pred).astype(int)

# F1
print("Micro F1:",    f1_score(y_true_bin, y_pred_bin, average="micro",    zero_division=0))
print("Macro F1:",    f1_score(y_true_bin, y_pred_bin, average="macro",    zero_division=0))
print("Weighted F1:", f1_score(y_true_bin, y_pred_bin, average="weighted", zero_division=0))

# Accuracy
hamming_acc = 1.0 - hamming_loss(y_true_bin, y_pred_bin)                                    # Hamming accuracy
jaccard_acc = jaccard_score(y_true_bin, y_pred_bin, average="samples", zero_division=0)     # Example-based Jaccard
subset_acc  = accuracy_score(y_true_bin, y_pred_bin)                                        # Subset accuracy

print("\nAccuracy metrics (multi-label):")
print(f"Hamming accuracy:            {hamming_acc:.4f}")
print(f"Example-based Jaccard acc.:  {jaccard_acc:.4f}")
print(f"Subset accuracy (exact match): {subset_acc:.4f}")

print("\nClassification Report:")
print(classification_report(
    y_true_bin, y_pred_bin,
    target_names=entity_vocab,
    zero_division=0
))

Micro F1: 0.7168709865732633
Macro F1: 0.5914609778422554
Weighted F1: 0.7242969091472737

Accuracy metrics (multi-label):
Hamming accuracy:            0.9712
Example-based Jaccard acc.:  0.4709
Subset accuracy (exact match): 0.4190

Classification Report:
                           precision    recall  f1-score   support

          Federal Reserve       0.88      0.80      0.84        87
           Interest Rates       0.69      0.82      0.75        50
                Inflation       0.89      0.86      0.87        93
               Employment       0.69      0.81      0.75        31
             Unemployment       1.00      1.00      1.00         8
                      GDP       0.60      0.68      0.64        31
                    Trade       1.00      1.00      1.00         3
                 Congress       0.50      0.33      0.40         3
          Monetary Policy       0.61      0.74      0.67        66
      Financial Stability       0.00      0.00      0.00         1
     

In [ ]:
# --- Per-entity metrics ---

N, L = y_true_bin.shape
rows = []
for i, name in enumerate(entity_vocab):
    yt = y_true_bin[:, i]
    yp = y_pred_bin[:, i]

    tp = int(np.sum((yt == 1) & (yp == 1)))
    tn = int(np.sum((yt == 0) & (yp == 0)))
    fp = int(np.sum((yt == 0) & (yp == 1)))
    fn = int(np.sum((yt == 1) & (yp == 0)))

    # per-entity Accuracy（Hamming accuracy）
    acc_i = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else np.nan

    # per-entity Jaccard（binary）
    jac_i = jaccard_score(yt, yp, average="binary", zero_division=0)

    # Precision / Recall / F1（binary）
    prec_i, rec_i, f1_i, supp_pos = precision_recall_fscore_support(
        yt, yp, average="binary", zero_division=0
    )

    rows.append({
        "entity": name,
        "support_total": int(len(yt)),
        "support_pos": int(np.sum(yt == 1)),
        "prevalence": float(np.mean(yt == 1)),
        "predicted_pos": int(np.sum(yp == 1)),
        "pred_rate": float(np.mean(yp == 1)),
        "accuracy": float(acc_i),
        "jaccard": float(jac_i),
        "precision": float(prec_i),
        "recall": float(rec_i),
        "f1": float(f1_i),
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
    })

per_entity_df = pd.DataFrame(rows)

# Jaccard
per_entity_df = per_entity_df.sort_values(["support_pos", "jaccard"], ascending=[False, False])

print("\nPer-entity metrics (support_pos then jaccard):")
print(per_entity_df.to_string(index=False))

# Across entities
macro_label_acc = per_entity_df["accuracy"].mean()
macro_label_jac = per_entity_df["jaccard"].mean()
macro_label_f1  = per_entity_df["f1"].mean()

print("\nMacro (over entities):")
print(f"Macro label accuracy: {macro_label_acc:.4f}")
print(f"Macro label Jaccard:  {macro_label_jac:.4f}")
print(f"Macro label F1:       {macro_label_f1:.4f}")


Per-entity metrics (support_pos then jaccard):
                   entity  support_total  support_pos  prevalence  predicted_pos  pred_rate  accuracy  jaccard  precision   recall       f1  TP  FP  FN  TN
         Economic Outlook            401          121    0.301746            157   0.391521  0.820449 0.588571   0.656051 0.851240 0.741007 103  54  18 226
                Inflation            401           93    0.231920             90   0.224439  0.942643 0.776699   0.888889 0.860215 0.874317  80  10  13 298
          Federal Reserve            401           87    0.216958             80   0.199501  0.932668 0.721649   0.875000 0.804598 0.838323  70  10  17 304
          Monetary Policy            401           66    0.164589             80   0.199501  0.880299 0.505155   0.612500 0.742424 0.671233  49  31  17 304
           Interest Rates            401           50    0.124688             59   0.147132  0.932668 0.602941   0.694915 0.820000 0.752294  41  18   9 333
                

# Evaluation Sentiment Analysis

In [ ]:
# Evaluate sentiment scoring (NLP and LLM) on the holdout set using gold entities.
# This script loads the holdout set, parses gold entity/sentiment annotations,
# runs both prediction methods, and computes evaluation metrics.

HOLDOUT_CSV = "data/holdout_eval_set.csv"

# Parse the "Entities" cell into a list of (entity, score) tuples
def parse_entities_cell(s):
    """Parse string representation of entity list into (entity, score) tuples."""
    if pd.isna(s) or s.strip() == "":
        return []
    try:
        raw = ast.literal_eval(s.strip())
    except Exception as e:
        print(f"Warning: Could not parse entities: {s[:50]}... Error: {e}")
        return []
    entities = []
    if not isinstance(raw, list):
        return []
    for item in raw:
        if not isinstance(item, (list, tuple)) or len(item) != 2:
            continue
        entity_name, score = item[0], item[1]
        if not entity_name or entity_name.strip() == "":
            continue
        try:
            score = float(score)
            entities.append((str(entity_name).strip(), score))
        except (ValueError, TypeError):
            print(f"Warning: Invalid score for entity '{entity_name}': {score}")
            continue
    return entities

# Load and prepare holdout data, returning a DataFrame of gold annotations
def load_holdout_data(csv_path):
    print(f"Loading holdout data from {csv_path}")
    df = pd.read_csv(csv_path)
    print(f"Raw CSV shape: {df.shape}")
    expected_cols = ['Sentence', 'Entities']
    if not all(col in df.columns for col in expected_cols):
        print(f"Warning: Expected columns {expected_cols}, got {list(df.columns)}")
    df = df.dropna(subset=['Sentence']).copy()
    df['Sentence'] = df['Sentence'].str.strip()
    df = df[df['Sentence'].str.len() > 0].copy()
    print(f"After cleaning: {len(df)} sentences")
    gold_data = []
    for idx, row in df.iterrows():
        sentence = row['Sentence']
        entities = parse_entities_cell(row.get('Entities', '[]'))
        if not entities:
            continue
        for pos, (entity_name, true_score) in enumerate(entities):
            gold_data.append({
                'sentence': sentence,
                'position': pos,
                'entity': entity_name,
                'true_score': true_score,
                'source_row': idx
            })
    gold_df = pd.DataFrame(gold_data)
    print(f"Gold standard: {len(gold_df)} entity annotations across {gold_df['sentence'].nunique()} sentences")
    return gold_df

# Prepare input formats for both NLP and LLM methods
def prepare_prediction_inputs(gold_df):
    # Group by sentence to get unique sentences and their entities
    sentence_groups = gold_df.groupby('sentence').apply(
        lambda x: x.sort_values('position')[['entity', 'true_score']].values.tolist()
    ).to_dict()
    print(f"Preparing inputs for {len(sentence_groups)} unique sentences")
    # LLM input: list of dicts with sentence and entities
    llm_inputs = []
    for sentence, entity_data in sentence_groups.items():
        entities = [{"name": ent} for ent, _ in entity_data]
        llm_inputs.append({
            "sentence": sentence,
            "entities": entities
        })
    # NLP input: list of (sentence, [entity names])
    nlp_inputs = []
    for sentence, entity_data in sentence_groups.items():
        entity_names = [ent for ent, _ in entity_data]
        nlp_inputs.append((sentence, entity_names))
    return llm_inputs, nlp_inputs

# Convert prediction results to a standardized DataFrame
def predictions_to_dataframe(predictions, method_name):
    rows = []
    if method_name == "LLM":
        for item in predictions:
            sentence = item["sentence"]
            for pos, entity_data in enumerate(item["entities"]):
                rows.append({
                    'sentence': sentence,
                    'position': pos,
                    'entity': entity_data["name"],
                    'pred_score': entity_data["sentiment"]
                })
    elif method_name == "NLP":
        for item in predictions:
            sentence = item["sentence"]
            for pos, entity_data in enumerate(item["entities"]):
                rows.append({
                    'sentence': sentence,
                    'position': pos,
                    'entity': entity_data["name"],
                    'pred_score': entity_data["sentiment"]
                })
    df = pd.DataFrame(rows)
    print(f"{method_name} predictions: {len(df)} entity predictions")
    return df

# Compute and print evaluation metrics for a prediction method
def compute_metrics(method_name, gold_df, pred_df, score_to_label):
    print(f"\n{'='*50}")
    print(f"EVALUATING: {method_name}")
    print(f"{'='*50}")
    merged = gold_df.merge(
        pred_df, 
        on=['sentence', 'position', 'entity'], 
        how='left',
        suffixes=('', '_pred')
    )
    print(f"Gold annotations: {len(gold_df)}")
    print(f"Predictions: {len(pred_df)}")
    print(f"Matched: {len(merged[~merged['pred_score'].isna()])}")
    print(f"Unmatched: {len(merged[merged['pred_score'].isna()])}")
    valid = merged[
        (~merged['pred_score'].isna()) & 
        (merged['pred_score'] != "") &
        (merged['pred_score'] != "")
    ].copy()
    if len(valid) == 0:
        print("No valid predictions to evaluate!")
        return
    def score_to_class(score):
        try:
            return score_to_label.get(float(score), None)
        except:
            return None
    valid['true_label'] = valid['true_score'].apply(score_to_class)
    valid['pred_label'] = valid['pred_score'].apply(score_to_class)
    valid_labels = valid[
        (~valid['true_label'].isna()) & 
        (~valid['pred_label'].isna())
    ].copy()
    if len(valid_labels) == 0:
        print("No valid label pairs after classification mapping!")
        return
    print(f"Valid for classification: {len(valid_labels)}")
    y_true = valid_labels['true_label'].astype(int).values
    y_pred = valid_labels['pred_label'].astype(int).values
    labels = sorted(set(score_to_label.values()))
    accuracy = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro', labels=labels, zero_division=0)
    micro_f1 = f1_score(y_true, y_pred, average='micro', labels=labels, zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, average='weighted', labels=labels, zero_division=0)
    y_true_reg = valid['true_score'].astype(float).values
    y_pred_reg = valid['pred_score'].astype(float).values
    mae = mean_absolute_error(y_true_reg, y_pred_reg)
    majority_score = pd.Series(y_true_reg).mode().iloc[0]
    median_score = np.median(y_true_reg)
    mae_majority = mean_absolute_error(y_true_reg, [majority_score] * len(y_true_reg))
    mae_median = mean_absolute_error(y_true_reg, [median_score] * len(y_true_reg))
    print(f"\n--- OVERALL METRICS (n={len(valid_labels)}) ---")
    print(f"Accuracy:           {accuracy:.4f}")
    print(f"Macro F1:           {macro_f1:.4f}")
    print(f"Micro F1:           {micro_f1:.4f}")
    print(f"Weighted F1:        {weighted_f1:.4f}")
    print(f"MAE:                {mae:.4f}")
    print(f"MAE (majority={majority_score:.2f}): {mae_majority:.4f} (Δ={mae_majority-mae:+.4f})")
    print(f"MAE (median={median_score:.2f}):   {mae_median:.4f} (Δ={mae_median-mae:+.4f})")
    print(f"\n--- PER-ENTITY BREAKDOWN ---")
    entity_stats = []
    for entity in sorted(valid_labels['entity'].unique()):
        entity_data = valid_labels[valid_labels['entity'] == entity]
        if len(entity_data) < 2:
            continue
        ent_y_true = entity_data['true_label'].astype(int).values
        ent_y_pred = entity_data['pred_label'].astype(int).values
        ent_acc = accuracy_score(ent_y_true, ent_y_pred)
        ent_f1 = f1_score(ent_y_true, ent_y_pred, average='macro', labels=labels, zero_division=0)
        ent_precision = precision_score(ent_y_true, ent_y_pred, average='macro', labels=labels, zero_division=0)
        ent_recall = recall_score(ent_y_true, ent_y_pred, average='macro', labels=labels, zero_division=0)
        entity_stats.append((entity, len(entity_data), ent_acc, ent_f1, ent_precision, ent_recall))
    # Sort by count descending, then F1 descending
    entity_stats.sort(key=lambda x: (-x[1], -x[3]))
    print(f"{'Entity':<25} {'n':>3}  {'acc':>6}  {'F1':>6}  {'Prec':>6}  {'Rec':>6}")
    for entity, count, acc, f1, prec, rec in entity_stats:
        print(f"{entity:<25} {count:3d}  {acc:6.3f}  {f1:6.3f}  {prec:6.3f}  {rec:6.3f}")
    print(f"\n--- CLASSIFICATION REPORT ---")
    print(classification_report(y_true, y_pred, labels=labels, zero_division=0))

# Main execution logic
def main():
    print("Starting Sentiment Evaluation")
    # Load score to label mapping
    try:
        with open("score_to_label.json", "r") as f:
            score_to_label_raw = json.load(f)
        score_to_label = {float(k): int(v) for k, v in score_to_label_raw.items()}
        print(f"Loaded score-to-label mapping: {len(score_to_label)} mappings")
    except FileNotFoundError:
        print("score_to_label.json not found!")
        return
    # Load holdout data
    try:
        gold_df = load_holdout_data(HOLDOUT_CSV)
    except FileNotFoundError:
        print(f"Holdout file not found: {HOLDOUT_CSV}")
        return
    if len(gold_df) == 0:
        print("No valid gold data found!")
        return
    # Prepare inputs for prediction methods
    llm_inputs, nlp_inputs = prepare_prediction_inputs(gold_df)
    # Run predictions
    print(f"\nRunning LLM predictions...")
    try:
        llm_predictions = extract_llm_sentiment(llm_inputs)
        llm_pred_df = predictions_to_dataframe(llm_predictions, "LLM")
    except Exception as e:
        print(f"LLM prediction failed: {e}")
        llm_pred_df = pd.DataFrame()
    print(f"\nRunning NLP predictions...")
    try:
        nlp_predictions = extract_nlp_sentiment(nlp_inputs)
        nlp_pred_df = predictions_to_dataframe(nlp_predictions, "NLP")
    except Exception as e:
        print(f"NLP prediction failed: {e}")
        nlp_pred_df = pd.DataFrame()
    # Evaluate both methods
    if not llm_pred_df.empty:
        compute_metrics("LLM", gold_df, llm_pred_df, score_to_label)
    if not nlp_pred_df.empty:
        compute_metrics("NLP", gold_df, nlp_pred_df, score_to_label)
    print(f"\nEvaluation complete!")

if __name__ == "__main__":
    main()

2025-08-31 14:13:11,422 | INFO | [extract] received items=284 | with_entities=284 | skipped=0
2025-08-31 14:13:11,423 | INFO | [extract] batching | chunks=15 | chunk_size≈20


Starting Sentiment Evaluation
Loaded score-to-label mapping: 7 mappings
Loading holdout data from data/holdout_eval_set.csv
Raw CSV shape: (400, 4)
After cleaning: 400 sentences
Gold standard: 717 entity annotations across 284 sentences
Preparing inputs for 284 unique sentences

Running LLM predictions...


2025-08-31 14:13:11,745 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-31 14:13:11,773 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-31 14:13:11,775 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-31 14:13:12,190 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-31 14:13:26,065 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-31 14:13:31,303 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-31 14:13:32,153 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-31 14:13:33,230 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-31 14:13:46,543 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 

LLM predictions: 717 entity predictions

Running NLP predictions...
[DEBUG] extract_nlp_sentiment called with 284 sentences
[DEBUG] Using threshold: 0.4
[DEBUG] Total entities to process: 717
[DEBUG] Sentence 1/284: 2 entities
[DEBUG]   Sentence text: '' Allow modest deviations from stated amounts for purchases and reinvestments, if needed for operati...'
[DEBUG]   Entities: ['Monetary Policy', 'Reinvestment']
[DEBUG]     Entity 1: 'Monetary Policy' -> canonical: 'Monetary Policy' (conf: 1.000)
[DEBUG]     Context: '' Allow modest deviations from stated amounts for purchases and reinvestments, i...'
[DEBUG]     Formatted: '[E] Monetary Policy [/E] ' Allow modest deviations from stated amounts for purchases and reinvestmen...'
[DEBUG]     Entity 2: 'Reinvestment' -> canonical: 'Reinvestment' (conf: 1.000)
[DEBUG]     Context: 'modest deviations from stated amounts for purchases and reinvestments, if needed...'
[DEBUG]     Formatted: '[E] Reinvestment [/E] modest deviations from stated a